In [3]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import os
import tensorflow as tf

2026-05-01 07:58:50.839220: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777622331.038852      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777622331.101275      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777622331.572500      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777622331.572544      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777622331.572547      57 computation_placer.cc:177] computation placer alr

In [2]:
!cp "/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part2/test/0001/01_01_0001_(10_11_16_16_20_30)_c.npy" /kaggle/working/

In [4]:
import os
import numpy as np
from sklearn.model_selection import train_test_split

base_dirs = [
    "/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part2",
    "/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part1",
    "/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part3"
]

def get_paths_and_labels(base_dirs, data_type):
    paths, labels = [], []
    unique_labels = sorted(os.listdir(os.path.join(base_dirs[0], "train")))
    label_to_index = {name: i for i, name in enumerate(unique_labels)}

    for base_dir in base_dirs:
        data_path = os.path.join(base_dir, data_type)
        if not os.path.exists(data_path): continue

        for label in unique_labels:
            label_dir = os.path.join(data_path, label)
            if os.path.exists(label_dir):
                for file in os.listdir(label_dir):
                    paths.append(os.path.join(label_dir, file))
                    labels.append(label_to_index[label])
    return paths, labels, len(unique_labels)

train_val_paths, train_val_labels, num_classes = get_paths_and_labels(base_dirs, "train")
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths,
    train_val_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_val_labels
)

test_paths, test_labels, _ = get_paths_and_labels(base_dirs, "test")

### Efficient Data Pre-processing: Converting to TFRecords
To solve the GPU bottleneck, we will convert the `.npy` files into `TFRecord` files. This allows TensorFlow to stream data directly from disk to the GPU without being blocked by the Python Global Interpreter Lock (GIL) or NumPy overhead.

In [5]:
import tensorflow as tf
import numpy as np
import tqdm

def _float_feature(value):
    return tf.train.Feature(float_list=tf.train.FloatList(value=value))

def _int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def create_tfrecord(paths, labels, output_file):
    with tf.io.TFRecordWriter(output_file) as writer:
        for path, label in tqdm.tqdm(zip(paths, labels), total=len(paths)):
            data = np.load(path).astype(np.float32)
            flattened_data = data.flatten().tolist()
            num_frames = data.shape[0]

            feature = {
                'data': _float_feature(flattened_data),
                'label': _int64_feature(label),
                'num_frames': _int64_feature(num_frames)
            }

            example = tf.train.Example(features=tf.train.Features(feature=feature))
            writer.write(example.SerializeToString())
if not os.path.exists('tfrecords/train.tfrecord'):
    os.makedirs('tfrecords', exist_ok=True)

    print("Converting Train set to TFRecords...")
    create_tfrecord(train_paths, train_labels, 'tfrecords/train.tfrecord')
    
    print("Converting Val set to TFRecords...")
    create_tfrecord(val_paths, val_labels, 'tfrecords/val.tfrecord')
    
    print("Converting Test set to TFRecords...")
    create_tfrecord(test_paths, test_labels, 'tfrecords/test.tfrecord')
else:
    print("TFRecords already exist. Skipping conversion.")

Converting Train set to TFRecords...


100%|██████████| 50774/50774 [06:37<00:00, 127.57it/s]


Converting Val set to TFRecords...


100%|██████████| 12694/12694 [01:59<00:00, 106.55it/s]


Converting Test set to TFRecords...


100%|██████████| 12046/12046 [01:53<00:00, 105.89it/s]


Now we define a fast loader that uses `tf.data.TFRecordDataset`. Note that since sequences have variable lengths, we still use `padded_batch`.

In [6]:
def parse_tfrecord_fn(example):
    feature_description = {
        'data': tf.io.VarLenFeature(tf.float32),
        'label': tf.io.FixedLenFeature([], tf.int64),
        'num_frames': tf.io.FixedLenFeature([], tf.int64),
    }
    example = tf.io.parse_single_example(example, feature_description)

    # Convert sparse tensor to dense and reshape
    data = tf.sparse.to_dense(example['data'])
    num_frames = tf.cast(example['num_frames'], tf.int32)
    data = tf.reshape(data, [num_frames, 126])

    label = tf.one_hot(tf.cast(example['label'], tf.int32), depth=num_classes)
    return data, label

def load_tfrecord_dataset(file_path, batch_size=32, shuffle=True):
    ds = tf.data.TFRecordDataset(file_path)
    if shuffle:
        ds = ds.shuffle(1000)
    ds = ds.map(parse_tfrecord_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.padded_batch(batch_size, padded_shapes=([None, 126], [num_classes]))
    return ds.prefetch(tf.data.AUTOTUNE)

strategy = tf.distribute.MirroredStrategy()
print(f"Number of devices: {strategy.num_replicas_in_sync}")

BATCH_SIZE_PER_REPLICA = 64
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync

train_ds = load_tfrecord_dataset('tfrecords/train.tfrecord', batch_size=GLOBAL_BATCH_SIZE, shuffle=True)
val_ds = load_tfrecord_dataset('tfrecords/val.tfrecord', batch_size=GLOBAL_BATCH_SIZE, shuffle=False)
test_ds = load_tfrecord_dataset('tfrecords/test.tfrecord', batch_size=GLOBAL_BATCH_SIZE, shuffle=False)

I0000 00:00:1777623083.971783      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777623083.977717      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of devices: 2


In [7]:
print(num_classes)

502


In [8]:
data = np.load("/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part2/test/0001/01_01_0001_(10_11_16_16_20_30)_c.npy")
print(f"Number of frames: {data.shape[0]}")
print(f"Number of landmarks: {data.shape[1]}")

Number of frames: 8
Number of landmarks: 126


In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import time
import numpy as np

class PositionalEncoding(layers.Layer):
    def __init__(self, d_model, max_steps=2000, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.pos_emb = layers.Embedding(input_dim=max_steps, output_dim=d_model)

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        return inputs + self.pos_emb(positions)

def build_temporal_transformer(num_landmarks, num_classes, d_model, num_heads, ff_dim, num_layers):
    inputs = layers.Input(shape=(None, num_landmarks))
    x = layers.Masking(mask_value=0.0)(inputs)

    x = layers.Dense(d_model)(x)
    x = PositionalEncoding(d_model)(x)

    for _ in range(num_layers):
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=0.1
        )(x, x)
        x = layers.Add()([x, attention_output])
        x = layers.LayerNormalization(epsilon=1e-6)(x)

        ffn = layers.Dense(ff_dim, activation="relu")(x)
        ffn = layers.Dropout(0.1)(ffn)
        ffn = layers.Dense(d_model)(ffn)

        x = layers.Add()([x, ffn])
        x = layers.LayerNormalization(epsilon=1e-6)(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(d_model // 2, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs=inputs, outputs=outputs)


In [28]:
# --- 2. Configuration & Data Setup ---
NUM_LANDMARKS = 126
NUM_CLASSES = 502
WINDOW_SIZE = 30 # Simulating a 30-frame rolling buffer

model_configs = {
    "Base (486k)":   {"d_model": 128, "num_heads": 4, "ff_dim": 256,  "num_layers": 1},
    "Medium (3.5M)": {"d_model": 256, "num_heads": 8, "ff_dim": 1024, "num_layers": 4},
    "Large (20M)":   {"d_model": 512, "num_heads": 8, "ff_dim": 2048, "num_layers": 6}
}

# Create dummy data for the benchmark (Replace with your train_ds in production)
print("Generating dummy data for initialization...")
dummy_x = tf.random.normal((128, WINDOW_SIZE, NUM_LANDMARKS))
dummy_y = tf.one_hot(tf.random.uniform((128,), maxval=NUM_CLASSES, dtype=tf.int32), depth=NUM_CLASSES)
dummy_dataset = tf.data.Dataset.from_tensor_slices((dummy_x, dummy_y)).batch(32)

# --- 3. The Benchmark Loop ---
results = {}

for name, config in model_configs.items():
    print(f"\n{'='*50}\nBenchmarking: {name}\n{'='*50}")
    
    # 1. Build and Compile
    model = build_temporal_transformer(
        num_landmarks=NUM_LANDMARKS,
        num_classes=NUM_CLASSES,
        **config
    )
    
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    # 2. Train for 1 Epoch (Initializes graph and weights)
    print("Training 1 warmup epoch to initialize computational graph...")
    model.fit(dummy_dataset, epochs=1, verbose=0)
    
    # 3. Setup Inference Benchmark
    # Wrap in tf.function to simulate optimized execution graph
    @tf.function(jit_compile=True) # jit_compile=True forces XLA optimization
    def optimized_predict(input_tensor):
        return model(input_tensor, training=False)

    # Simulate a single sliding-window frame: shape (1, 30, 126)
    single_frame_sequence = tf.random.normal((1, WINDOW_SIZE, NUM_LANDMARKS))
    
    print("Warming up inference engine...")
    for _ in range(10):
        _ = optimized_predict(single_frame_sequence)
        
    print("Running latency test (1000 iterations)...")
    iterations = 1000
    
    # Wait for GPU to finish any lingering operations before starting the clock
    tf.test.experimental.sync_devices()
    start_time = time.perf_counter()
    
    for _ in range(iterations):
        _ = optimized_predict(single_frame_sequence)
        
    # Wait for all GPU operations to finish before stopping the clock
    tf.test.experimental.sync_devices()
    end_time = time.perf_counter()
    
    # 4. Calculate Results
    total_time = end_time - start_time
    avg_latency_ms = (total_time / iterations) * 1000
    
    results[name] = avg_latency_ms
    print(f"✅ Result for {name}: {avg_latency_ms:.2f} milliseconds per frame")

# --- 4. Final Report ---
print("\n" + "="*50)
print("FINAL INFERENCE LATENCY REPORT (Target: < 33ms)")
print("="*50)
for name, latency in results.items():
    fps_equivalent = 1000 / latency if latency > 0 else 0
    print(f"{name.ljust(15)} : {latency:>5.2f} ms | Max capability: {fps_equivalent:>5.0f} FPS")

Generating dummy data for initialization...

Benchmarking: Base (486k)
Training 1 warmup epoch to initialize computational graph...


I0000 00:00:1777590571.318101     134 service.cc:152] XLA service 0x7ec61c002e40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777590571.318146     134 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777590571.318150     134 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777590572.010236     134 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777590576.375323     134 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Warming up inference engine...
Running latency test (1000 iterations)...
✅ Result for Base (486k): 0.48 milliseconds per frame

Benchmarking: Medium (3.5M)
Training 1 warmup epoch to initialize computational graph...
Warming up inference engine...
Running latency test (1000 iterations)...
✅ Result for Medium (3.5M): 0.74 milliseconds per frame

Benchmarking: Large (20M)
Training 1 warmup epoch to initialize computational graph...
Warming up inference engine...
Running latency test (1000 iterations)...
✅ Result for Large (20M): 1.10 milliseconds per frame

FINAL INFERENCE LATENCY REPORT (Target: < 33ms)
Base (486k)     :  0.48 ms | Max capability:  2091 FPS
Medium (3.5M)   :  0.74 ms | Max capability:  1360 FPS
Large (20M)     :  1.10 ms | Max capability:   910 FPS


In [31]:
from tensorflow import keras

NUM_LANDMARKS = 126
NUM_CLASSES = 502
WINDOW_SIZE = 30
with strategy.scope():
    landmark_transformer_model = build_temporal_transformer(
        num_landmarks=NUM_LANDMARKS,
        num_classes=NUM_CLASSES,
        **model_configs["Large (20M)"]
    )

    landmark_transformer_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=[keras.metrics.CategoricalAccuracy()]
    )

landmark_transformer_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, None, 126) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_3 (Masking) │ (None, None, 126) │          0 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_31 (Dense)    │ (None, None, 512) │     65,024 │ masking_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encodin… │ (None, None, 512) │  1,024,000 │ dense_31[0][0]    │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 512) │  1,050,624 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_22 (Add)        │ (None, None, 512) │          0 │ positional_encod… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 512) │      1,024 │ add_22[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_32 (Dense)    │ (None, None,      │  1,050,624 │ layer_normalizat… │
│                     │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_26          │ (None, None,      │          0 │ dense_32[0][0]    │
│ (Dropout)           │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_33 (Dense)    │ (None, None, 512) │  1,049,088 │ dropout_26[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_23 (Add)        │ (None, None, 512) │          0 │ layer_normalizat… │
│                     │                   │            │ dense_33[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 512) │      1,024 │ add_23[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 512) │  1,050,624 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_24 (Add)        │ (None, None, 512) │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 512) │      1,024 │ add_24[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_34 (Dense)    │ (None, None,      │  1,050,624 │ layer_normalizat… │
│                     │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_28          │ (None, None,      │          0 │ dense_34[0][0]    │
│ (Dropout)           │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_35 (Dense)    │ (None, None, 512) │  1,049,088 │ dropout_28[0][0]

 Total params: 20,263,670 (77.30 MB)

 Trainable params: 20,263,670 (77.30 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
history = landmark_transformer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ]
)

Epoch 1/10
397/397 ━━━━━━━━━━━━━━━━━━━━ 101s 255ms/step - categorical_accuracy: 0.9637 - loss: 1.2149 - val_categorical_accuracy: 0.9434 - val_loss: 1.2198 - learning_rate: 1.0000e-04
Epoch 2/10
397/397 ━━━━━━━━━━━━━━━━━━━━ 100s 252ms/step - categorical_accuracy: 0.9614 - loss: 1.2222 - val_categorical_accuracy: 0.9580 - val_loss: 1.1698 - learning_rate: 1.0000e-04
Epoch 3/10
397/397 ━━━━━━━━━━━━━━━━━━━━ 100s 252ms/step - categorical_accuracy: 0.9678 - loss: 1.1901 - val_categorical_accuracy: 0.9587 - val_loss: 1.1734 - learning_rate: 1.0000e-04
Epoch 4/10
397/397 ━━━━━━━━━━━━━━━━━━━━ 100s 251ms/step - categorical_accuracy: 0.9706 - loss: 1.1814 - val_categorical_accuracy: 0.9590 - val_loss: 1.1658 - learning_rate: 1.0000e-04
Epoch 5/10
397/397 ━━━━━━━━━━━━━━━━━━━━ 100s 251ms/step - categorical_accuracy: 0.9733 - loss: 1.1651 - val_categorical_accuracy: 0.9547 - val_loss: 1.1750 - learning_rate: 1.0000e-04
Epoch 6/10
397/397 ━━━━━━━━━━━━━━━━━━━━ 100s 251ms/step - categorical_accuracy: 

In [ ]:
import numpy as np
import time

# Assuming you have the RealTimeSignPredictor from the previous step initialized
# predictor = RealTimeSignPredictor("model_quantized.tflite", window_size=30)

def simulate_realtime_inference(npy_file_path, predictor_instance, class_names_map):
    """
    Simulates a real-time video feed by passing rows of a .npy file 
    into the sliding window predictor one by one.
    """
    print(f"Loading pre-processed video landmarks from: {npy_file_path}")
    
    # 1. Load the data
    try:
        # Load the float16 array and convert to float32 for the model
        video_landmarks = np.load(npy_file_path).astype(np.float32)
    except Exception as e:
        print(f"Failed to load file: {e}")
        return

    total_frames = video_landmarks.shape[0]
    print(f"Successfully loaded {total_frames} frames (126 landmarks each).")
    
    # Clear any residual data in the buffer before starting a new video
    predictor_instance.reset()
    
    print("\nStarting simulated stream...")
    print("-" * 40)
    
    # 2. Simulate frame-by-frame processing
    for frame_idx, frame_data in enumerate(video_landmarks):
        
        # Simulate the time delay of a real video stream (optional, for realism)
        # If the original video was 30fps and we skip half, effective is 15fps (~66ms per frame)
        # time.sleep(0.066) 
        
        # Pass the single frame (shape: 126,) to the sliding window
        prediction_id = predictor_instance.process_frame(frame_data)
        
        if prediction_id is not None:
            # Buffer is full, we got a prediction
            sign_word = class_names_map.get(prediction_id, f"Class {prediction_id}")
            print(f"Frame {frame_idx:03d} | Buffer Full | Prediction: {sign_word}")
        else:
            # Buffer is still filling up
            print(f"Frame {frame_idx:03d} | Filling Buffer ({len(predictor_instance.frames_buffer)}/{predictor_instance.window_size})...")

    print("-" * 40)
    print("Video stream complete.")

# 1. Define your path
test_video_path = "/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part1/test/0002/01_02_0002_(15_11_16_16_42_51)_c.npy"

# 2. Create a dummy class map for visualization (Replace with your actual label encoder mapping)
dummy_class_map = {i: f"Arabic_Sign_{i}" for i in range(502)}

# 3. Run the simulation
simulate_realtime_inference(test_video_path, predictor, dummy_class_map)

In [40]:
import numpy as np
from sklearn.metrics import top_k_accuracy_score

print("Extracting raw predictions for deep analysis...")

y_true = []
y_pred_probs = []
for x_batch, y_batch in test_ds:
    predictions = landmark_transformer_model.predict(x_batch, verbose=0)
    batch_y_true = np.argmax(y_batch.numpy(), axis=1)
    y_true.extend(batch_y_true)
    y_pred_probs.extend(predictions)

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

top1_acc = np.mean(y_true == y_pred_classes)

all_labels = np.arange(NUM_CLASSES) 
top5_acc = top_k_accuracy_score(y_true, y_pred_probs, k=5, labels=all_labels)

print("\n" + "="*40)
print("FINAL TEST METRICS")
print("="*40)
print(f"Top-1 Accuracy : {top1_acc * 100:.2f}% (Exact Match)")
print(f"Top-5 Accuracy : {top5_acc * 100:.2f}% (In top 5 guesses)")
incorrect_mask = y_true != y_pred_classes
wrong_true_labels = y_true[incorrect_mask]
wrong_pred_labels = y_pred_classes[incorrect_mask]

print("\n" + "-"*40)
print("Error Analysis: Most Common Mistakes")
print("-" * 40)
mistake_pairs = list(zip(wrong_true_labels, wrong_pred_labels))
unique_mistakes, counts = np.unique(mistake_pairs, axis=0, return_counts=True)

sorted_indices = np.argsort(-counts)
for i in range(min(5, len(sorted_indices))):
    idx = sorted_indices[i]
    true_class = unique_mistakes[idx][0]
    pred_class = unique_mistakes[idx][1]
    count = counts[idx]
    print(f"True Sign [{true_class}] was misclassified as [{pred_class}] -> {count} times")

Extracting raw predictions for deep analysis...

FINAL TEST METRICS
Top-1 Accuracy : 95.57% (Exact Match)
Top-5 Accuracy : 98.71% (In top 5 guesses)

----------------------------------------
Error Analysis: Most Common Mistakes
----------------------------------------
True Sign [27] was misclassified as [26] -> 16 times
True Sign [44] was misclassified as [45] -> 6 times
True Sign [13] was misclassified as [14] -> 6 times
True Sign [18] was misclassified as [17] -> 5 times
True Sign [207] was misclassified as [200] -> 5 times


In [15]:
pip install tf2onnx onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 13.4 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [41]:
import tf2onnx
import onnx

WINDOW_SIZE = 30
NUM_LANDMARKS = 126

input_signature = [
    tf.TensorSpec([None, WINDOW_SIZE, NUM_LANDMARKS], tf.float32, name='input_landmarks')
]

onnx_model, _ = tf2onnx.convert.from_keras(landmark_transformer_model, input_signature, opset=15)
onnx_file_path = "arabic_sign_model_large.onnx"
onnx.save(onnx_model, onnx_file_path)

I0000 00:00:1777595892.664260      57 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1777595892.664500      57 single_machine.cc:374] Starting new session
I0000 00:00:1777595892.678064      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777595892.679562      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1777595894.721864      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777595894.723387      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 M

In [11]:
NUM_LANDMARKS = 126
NUM_CLASSES = 502
with strategy.scope():
    small_model = build_temporal_transformer(
        num_landmarks=NUM_LANDMARKS,
        num_classes=NUM_CLASSES,
        d_model=128, 
        num_heads=4, 
        ff_dim=256,  
        num_layers=1
    )
    
    small_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=[keras.metrics.CategoricalAccuracy()]
    )

small_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 126) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking (Masking)   │ (None, None, 126) │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 128) │     16,256 │ masking[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoding │ (None, None, 128) │    256,000 │ dense[0][0]       │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 128) │     66,048 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None, 128) │          0 │ positional_encod… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, None, 128) │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 256) │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, None, 256) │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, None, 128) │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, None, 128) │          0 │ layer_normalizat… │
│                     │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 128) │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 502)       │     32,630 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 445,622 (1.70 MB)

 Trainable params: 445,622 (1.70 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
from sklearn.metrics import top_k_accuracy_score
history = small_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', 
            patience=5, 
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', 
            factor=0.5, 
            patience=3
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath='best_small_model.keras',
            monitor='val_categorical_accuracy',
            save_best_only=True
        )
    ]
)

print("\n--- Running Final Evaluation ---")
test_loss, test_accuracy = small_model.evaluate(test_ds)
print(f"Base Test Loss: {test_loss:.4f}")

print("\nExtracting detailed predictions for Top-5 and Error Analysis...")
y_true = []
y_pred_probs = []

for x_batch, y_batch in test_ds:
    predictions = small_model.predict(x_batch, verbose=0)
    batch_y_true = np.argmax(y_batch.numpy(), axis=1)
    
    y_true.extend(batch_y_true)
    y_pred_probs.extend(predictions)

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

# Metrics
top1_acc = np.mean(y_true == y_pred_classes)
all_labels = np.arange(NUM_CLASSES)
top5_acc = top_k_accuracy_score(y_true, y_pred_probs, k=5, labels=all_labels)

print("\n========================================")
print("FINAL TEST METRICS (Base Model)")
print("========================================")
print(f"Top-1 Accuracy : {top1_acc * 100:.2f}%")
print(f"Top-5 Accuracy : {top5_acc * 100:.2f}%")

# Error Analysis
incorrect_mask = y_true != y_pred_classes
wrong_true_labels = y_true[incorrect_mask]
wrong_pred_labels = y_pred_classes[incorrect_mask]

print("\n----------------------------------------")
print("Error Analysis: Top 5 Confused Signs")
print("----------------------------------------")
if len(wrong_true_labels) > 0:
    mistake_pairs = list(zip(wrong_true_labels, wrong_pred_labels))
    unique_mistakes, counts = np.unique(mistake_pairs, axis=0, return_counts=True)
    sorted_indices = np.argsort(-counts)

    for i in range(min(5, len(sorted_indices))):
        idx = sorted_indices[i]
        true_class = unique_mistakes[idx][0]
        pred_class = unique_mistakes[idx][1]
        count = counts[idx]
        print(f"True Sign [{true_class}] was misclassified as [{pred_class}] -> {count} times")
else:
    print("Zero errors made on the test set!")

Epoch 1/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - categorical_accuracy: 0.8714 - loss: 1.8653 - val_categorical_accuracy: 0.9401 - val_loss: 1.5525 - learning_rate: 2.5000e-05
Epoch 2/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - categorical_accuracy: 0.8734 - loss: 1.8570 - val_categorical_accuracy: 0.9411 - val_loss: 1.5500 - learning_rate: 2.5000e-05
Epoch 3/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - categorical_accuracy: 0.8688 - loss: 1.8605 - val_categorical_accuracy: 0.9401 - val_loss: 1.5523 - learning_rate: 2.5000e-05
Epoch 4/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - categorical_accuracy: 0.8711 - loss: 1.8553 - val_categorical_accuracy: 0.9397 - val_loss: 1.5489 - learning_rate: 2.5000e-05
Epoch 5/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - categorical_accuracy: 0.8707 - loss: 1.8578 - val_categorical_accuracy: 0.9415 - val_loss: 1.5498 - learning_rate: 2.5000e-05
Epoch 6/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - categorical_accuracy: 0.8735 - los

In [19]:
import tf2onnx
import onnx

WINDOW_SIZE = 30
NUM_LANDMARKS = 126

input_signature = [
    tf.TensorSpec([None, WINDOW_SIZE, NUM_LANDMARKS], tf.float32, name='input_landmarks')
]

onnx_model, _ = tf2onnx.convert.from_keras(small_model, input_signature, opset=15)
onnx_file_path = "arabic_sign_model_small.onnx"
onnx.save(onnx_model, onnx_file_path)

I0000 00:00:1777625570.822355      57 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1777625570.822691      57 single_machine.cc:374] Starting new session
I0000 00:00:1777625570.837124      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777625570.838656      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1777625571.016853      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777625571.018398      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 M